In [1]:
# voting
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

In [2]:
# 1. Generate a synthetic dataset
X, y = make_classification(n_samples=500, n_features=20, n_informative=5, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [6]:
# Voting classifier
# 2. Define individual base estimators
# Note: SVC needs probability=True enabled to be used in Soft Voting



clf1 = LogisticRegression(random_state=42)
clf2 = DecisionTreeClassifier(random_state=42,max_depth=3)
clf3 = SVC(random_state=42)

In [7]:
hard_voting_clf = VotingClassifier(
    estimators=[('lr', clf1), ('dt', clf2), ('svc', clf3)],
    voting='hard'
)

In [8]:
hard_voting_clf.fit(X_train, y_train)
hard_preds = hard_voting_clf.predict(X_test)
print(f"Hard Voting Accuracy: {accuracy_score(y_test, hard_preds):.4f}")


Hard Voting Accuracy: 0.8333


In [11]:
# Regression
# 1. Generate a synthetic dataset


In [21]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Generate a synthetic regression dataset
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Define individual base estimators
reg1 = LinearRegression()
reg2 = DecisionTreeRegressor(max_depth=5, random_state=42)
reg3 = SVR(kernel='rbf')

# 3. Implement the Voting Regressor
# You can optionally pass weights to favor a stronger model, e.g., weights=[1, 1, 2]
voting_reg = VotingRegressor(
    estimators=[('lr', reg1), ('dt', reg2), ('svr', reg3)],
    weights=[1, 1, 1]  # Equal weight to all models by default
)

# 4. Train the ensemble
voting_reg.fit(X_train, y_train)

# 5. Evaluate the ensemble
preds = voting_reg.predict(X_test)
print(f"Voting Regressor R² Score: {r2_score(y_test, preds):.4f}")
print(f"Voting Regressor MSE: {mean_squared_error(y_test, preds):.4f}")

Voting Regressor R² Score: 0.7911
Voting Regressor MSE: 3872.3010


In [22]:
# Stacking

In [33]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# 1. Generate a complex, noisy dataset (1,000 samples, 20 features)
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_clusters_per_class=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Define base estimators
base_models = [
    ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
    ("svc", SVC( random_state=42)),
]

# 3. Define Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=base_models, final_estimator=LogisticRegression(), cv=5
)

# 4. Train and evaluate
stacking_clf.fit(X_train, y_train)
accuracy = stacking_clf.score(X_test, y_test)
print(f"Realistic Stacked Model Accuracy: {accuracy:.4f}")


Realistic Stacked Model Accuracy: 0.9400


In [34]:
# Stacking Regressor


In [35]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor

# 1. Load data
X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Define base estimators (Level 0)
# It works best when combining completely different types of algorithms
base_models = [
    ("rf", RandomForestRegressor(n_estimators=50, random_state=42)),
    ("mlp", MLPRegressor(max_iter=500, random_state=42)),
]

# 3. Define the Stacking Regressor with a Meta-Model (Level 1)
stacking_reg = StackingRegressor(
    estimators=base_models,
    final_estimator=RidgeCV(),  # Default meta-model for regression
    cv=5,  # 5-fold cross-validation to train the meta-model
)

# 4. Train and evaluate
stacking_reg.fit(X_train, y_train)
r2_score = stacking_reg.score(X_test, y_test)
print(f"Stacked Model R² Score: {r2_score:.4f}")


Stacked Model R² Score: 0.8046
